# Step 4: Daily Boarding Prediction (Baselines & ML Regressor)

## Objectives
1. Load chronological demand feature splits generated in Step 3[cite: 1].
2. Evaluate a **Seasonal Baseline** ($Lag_7, Lag_{14}, Lag_{21}, Lag_{28}$ mean) on the Validation set[cite: 1].
3. Train a **Random Forest Regressor** using calendar features, route encodings, and historical lags[cite: 1].
4. Compare performance across Validation and Test sets using **MAE**, **RMSE**, and **WAPE**[cite: 1].
5. Save the trained model pipeline artifact for the backend API[cite: 1].

In [1]:
import os
import pickle
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed")
MODEL_OUTPUT_PATH = os.path.join("..", "backend", "models")
os.makedirs(MODEL_OUTPUT_PATH, exist_ok=True)

# 1. Load feature dataset
df = pd.read_parquet(os.path.join(PROCESSED_DATA_PATH, "demand_features_2yr.parquet"))

# Enforce route categorical encoding via frequency/target counts
route_freq = df['route'].value_counts().to_dict()
df['route_encoded'] = df['route'].map(route_freq)

print(f"Dataset Loaded: {len(df):,} total rows")

Dataset Loaded: 83,997 total rows


## Define Metric Calculations & Baseline Evaluator
We calculate MAE, RMSE, and WAPE:
$$WAPE = \frac{\sum |y - \hat{y}|}{\sum y} \times 100$$

In [2]:
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) * 100
    return {"MAE": round(mae, 2), "RMSE": round(rmse, 2), "WAPE (%)": round(wape, 2)}

# Split dataset back into Train, Val, and Test
train_df = df[df['split'] == 'train'].copy()
val_df   = df[df['split'] == 'val'].copy()
test_df  = df[df['split'] == 'test'].copy()

# Evaluate Seasonal Baseline on Validation Set (only rows with available seasonal mean)
val_baseline = val_df.dropna(subset=['seasonal_mean', 'rides'])
baseline_metrics = calculate_metrics(val_baseline['rides'], val_baseline['seasonal_mean'])

print("================ SEASONAL BASELINE METRICS (VAL SET) ================")
print(f"Evaluated Samples : {len(val_baseline):,}")
print(f"MAE              : {baseline_metrics['MAE']}")
print(f"RMSE             : {baseline_metrics['RMSE']}")
print(f"WAPE             : {baseline_metrics['WAPE (%)']}%")
print("=====================================================================")

================ SEASONAL BASELINE METRICS (VAL SET) ================
Evaluated Samples : 10,222
MAE              : 423.66
RMSE             : 681.29
WAPE             : 8.61%


## Train Random Forest Regressor & Evaluate
Train model on features ($Lag_1, Lag_7, Lag_{14}, Lag_{21}, Lag_{28}$, day_of_week, month, is_weekend, route_encoded)[cite: 1].

In [3]:
feature_cols = ['lag_1', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'day_of_week', 'month', 'is_weekend', 'route_encoded']

# Impute missing lags with 0 for tree-based regressor
X_train = train_df[feature_cols].fillna(0)
y_train = train_df['rides']

X_val = val_df[feature_cols].fillna(0)
y_val = val_df['rides']

X_test = test_df[feature_cols].fillna(0)
y_test = test_df['rides']

# Train Random Forest Regressor
print("Training Random Forest Regressor...")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Generate predictions
val_preds = rf_model.predict(X_val)
test_preds = rf_model.predict(X_test)

val_rf_metrics = calculate_metrics(y_val, val_preds)
test_rf_metrics = calculate_metrics(y_test, test_preds)

print("\n================ MODEL EVALUATION SUMMARY ================")
print(f"Val Set RF  -> MAE: {val_rf_metrics['MAE']}, RMSE: {val_rf_metrics['RMSE']}, WAPE: {val_rf_metrics['WAPE (%)']}%")
print(f"Test Set RF -> MAE: {test_rf_metrics['MAE']}, RMSE: {test_rf_metrics['RMSE']}, WAPE: {test_rf_metrics['WAPE (%)']}%")
print("==========================================================")

# Save trained artifact
model_filepath = os.path.join(MODEL_OUTPUT_PATH, "rf_demand_model.pkl")
with open(model_filepath, "wb") as f:
    pickle.dump({"model": rf_model, "features": feature_cols, "route_encoding": route_freq}, f)

print(f"\nTrained model successfully saved to: {model_filepath}")

Training Random Forest Regressor...

================ MODEL EVALUATION SUMMARY ================
Val Set RF  -> MAE: 327.16, RMSE: 538.71, WAPE: 6.66%
Test Set RF -> MAE: 360.23, RMSE: 644.36, WAPE: 7.24%

Trained model successfully saved to: ..\backend\models\rf_demand_model.pkl


##XGBoost and light GBM model comparision

In [4]:
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import GradientBoostingRegressor

# Dictionary to hold models and their metric results
models = {
    "Random Forest": rf_model,
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, random_state=42, learning_rate=0.05, max_depth=6),
    "LightGBM": lgb.LGBMRegressor(n_estimators=100, random_state=42, learning_rate=0.05, verbose=-1)
}

benchmark_results = []

print("Running multi-model benchmark comparison...\n")

for name, model in models.items():
    if name != "Random Forest":
        model.fit(X_train, y_train)
    
    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)
    
    val_m = calculate_metrics(y_val, val_pred)
    test_m = calculate_metrics(y_test, test_pred)
    
    benchmark_results.append({
        "Model": name,
        "Val MAE": val_m["MAE"],
        "Val RMSE": val_m["RMSE"],
        "Val WAPE (%)": val_m["WAPE (%)"],
        "Test MAE": test_m["MAE"],
        "Test RMSE": test_m["RMSE"],
        "Test WAPE (%)": test_m["WAPE (%)"]
    })

comparison_df = pd.DataFrame(benchmark_results).sort_values(by="Val WAPE (%)")

print("=================== MODEL BENCHMARK RESULTS ===================")
print(comparison_df.to_string(index=False))
print("===============================================================")

# Save the best performing model based on Validation WAPE
best_model_name = comparison_df.iloc[0]["Model"]
best_model = models[best_model_name]

best_model_filepath = os.path.join(MODEL_OUTPUT_PATH, "best_demand_model.pkl")
with open(best_model_filepath, "wb") as f:
    pickle.dump({"model": best_model, "name": best_model_name, "features": feature_cols, "route_encoding": route_freq}, f)

print(f"\nBest Model selected: '{best_model_name}' saved to: {best_model_filepath}")

Running multi-model benchmark comparison...

=================== MODEL BENCHMARK RESULTS ===================
            Model  Val MAE  Val RMSE  Val WAPE (%)  Test MAE  Test RMSE  Test WAPE (%)
          XGBoost   324.64    519.03          6.61    366.38     643.42           7.37
    Random Forest   327.16    538.71          6.66    360.23     644.36           7.24
         LightGBM   331.93    528.34          6.76    370.26     641.19           7.44
Gradient Boosting   348.28    550.25          7.09    386.18     661.42           7.76

Best Model selected: 'XGBoost' saved to: ..\backend\models\best_demand_model.pkl


In [6]:
from sklearn.metrics import r2_score

r2_benchmark_results = []

for name, model in models.items():
    # Generate predictions across all 3 splits
    train_pred = model.predict(X_train)
    val_pred   = model.predict(X_val)
    test_pred  = model.predict(X_test)
    
    # Calculate R2 scores (multiplied by 100 for percentage representation)
    train_r2 = r2_score(y_train, train_pred) * 100
    val_r2   = r2_score(y_val, val_pred) * 100
    test_r2  = r2_score(y_test, test_pred) * 100
    
    r2_benchmark_results.append({
        "Model": name,
        "Train R² (%)": round(train_r2, 2),
        "Val R² (%)": round(val_r2, 2),
        "Test R² (%)": round(test_r2, 2)
    })

r2_df = pd.DataFrame(r2_benchmark_results).sort_values(by="Val R² (%)", ascending=False)

print("=================== MODEL R² SCORE ACCURACY (%) ===================")
print(r2_df.to_string(index=False))
print("===================================================================")

=================== MODEL R² SCORE ACCURACY (%) ===================
            Model  Train R² (%)  Val R² (%)  Test R² (%)
          XGBoost         97.82       98.73        98.12
         LightGBM         97.65       98.68        98.13
    Random Forest         99.57       98.63        98.11
Gradient Boosting         97.01       98.57        98.01
